# 📊 Análise de Performance — Trade Journal
XAU/USD · SMC/ICT Strategy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('dark_background')
GOLD = '#C9973A'
GREEN = '#1DB954'
RED = '#E03E3E'
BLUE = '#3B82F6'

df = pd.read_csv('../data/trades.csv')
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').reset_index(drop=True)
print(f'Total de trades: {len(df)}')
df.head()

## 1. Estatísticas Gerais

In [ ]:
total = len(df)
wins = len(df[df['resultado']=='WIN'])
losses = len(df[df['resultado']=='LOSS'])
bes = len(df[df['resultado']=='BE'])
wr = (wins/total*100)
total_r = df['rr'].sum()
avg_rr = df[df['resultado']=='WIN']['rr'].mean()

print(f'Total de Trades : {total}')
print(f'Wins            : {wins}')
print(f'Losses          : {losses}')
print(f'Break Evens     : {bes}')
print(f'Win Rate        : {wr:.1f}%')
print(f'Resultado Total : {total_r:.1f}R')
print(f'RR Médio (WIN)  : {avg_rr:.2f}R')

## 2. Curva de Capital

In [ ]:
df['r_acumulado'] = df['rr'].cumsum()

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(df['data'], df['r_acumulado'], color=GREEN, lw=2)
ax.fill_between(df['data'], df['r_acumulado'], alpha=0.15, color=GREEN)
ax.axhline(0, color='white', lw=0.5, alpha=0.3)
ax.set_title('Curva de Capital (R Acumulado)', color=GOLD, fontsize=13)
ax.set_ylabel('R', color='white')
ax.tick_params(colors='gray')
ax.set_facecolor('#0E1318')
fig.patch.set_facecolor('#090C10')
plt.tight_layout()
plt.savefig('../images/curva_capital.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Performance por Sessão

In [ ]:
sessao = df.groupby('sessao').agg(
    trades=('rr','count'),
    resultado_r=('rr','sum'),
    win_rate=('resultado', lambda x: (x=='WIN').mean()*100)
).round(2)

fig, axes = plt.subplots(1,2, figsize=(12,4))
cores = [GREEN if v>=0 else RED for v in sessao['resultado_r']]

sessao['resultado_r'].plot(kind='bar', ax=axes[0], color=cores, edgecolor='none')
axes[0].set_title('Resultado por Sessão (R)', color=GOLD)
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15, colors='white')
axes[0].tick_params(axis='y', colors='gray')
axes[0].set_facecolor('#0E1318')

sessao['win_rate'].plot(kind='bar', ax=axes[1], color=BLUE, edgecolor='none')
axes[1].set_title('Win Rate por Sessão (%)', color=GOLD)
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15, colors='white')
axes[1].tick_params(axis='y', colors='gray')
axes[1].set_facecolor('#0E1318')

for ax in axes: ax.figure.patch.set_facecolor('#090C10')
plt.tight_layout()
plt.savefig('../images/performance_sessao.png', dpi=150, bbox_inches='tight')
plt.show()
print(sessao)

## 4. Performance por Gatilho

In [ ]:
gatilho = df.groupby('gatilho').agg(
    trades=('rr','count'),
    resultado_r=('rr','sum'),
    win_rate=('resultado', lambda x: (x=='WIN').mean()*100)
).round(2)

fig, axes = plt.subplots(1,2, figsize=(12,4))
cores = [GREEN if v>=0 else RED for v in gatilho['resultado_r']]

gatilho['resultado_r'].plot(kind='bar', ax=axes[0], color=cores, edgecolor='none')
axes[0].set_title('Resultado por Gatilho (R)', color=GOLD)
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0, colors='white')
axes[0].tick_params(axis='y', colors='gray')
axes[0].set_facecolor('#0E1318')

gatilho['win_rate'].plot(kind='bar', ax=axes[1], color=BLUE, edgecolor='none')
axes[1].set_title('Win Rate por Gatilho (%)', color=GOLD)
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0, colors='white')
axes[1].tick_params(axis='y', colors='gray')
axes[1].set_facecolor('#0E1318')

for ax in axes: ax.figure.patch.set_facecolor('#090C10')
plt.tight_layout()
plt.savefig('../images/performance_gatilho.png', dpi=150, bbox_inches='tight')
plt.show()
print(gatilho)

## 5. Impacto do Estado Emocional

In [ ]:
emocao = df.groupby('emocao').agg(
    trades=('rr','count'),
    resultado_r=('rr','sum'),
    win_rate=('resultado', lambda x: (x=='WIN').mean()*100)
).round(2).sort_values('resultado_r', ascending=False)

fig, ax = plt.subplots(figsize=(12,4))
cores = [GREEN if v>=0 else RED for v in emocao['resultado_r']]
emocao['resultado_r'].plot(kind='bar', ax=ax, color=cores, edgecolor='none')
ax.set_title('Resultado por Estado Emocional (R)', color=GOLD, fontsize=13)
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=15, colors='white')
ax.tick_params(axis='y', colors='gray')
ax.set_facecolor('#0E1318')
fig.patch.set_facecolor('#090C10')
plt.tight_layout()
plt.savefig('../images/performance_emocao.png', dpi=150, bbox_inches='tight')
plt.show()
print(emocao)

## 6. Passo mais Forçado (Trades com Erro)

In [ ]:
erros = df[df['passo_forcado'] != 'Nenhum'].copy()
erros['passo_forcado'] = erros['passo_forcado'].astype(str)
contagem = erros['passo_forcado'].value_counts()

fig, ax = plt.subplots(figsize=(8,4))
contagem.plot(kind='bar', ax=ax, color=RED, edgecolor='none')
ax.set_title('Passo mais Forçado nos Trades com Erro', color=GOLD, fontsize=13)
ax.set_xlabel('Passo')
ax.tick_params(axis='x', rotation=0, colors='white')
ax.tick_params(axis='y', colors='gray')
ax.set_facecolor('#0E1318')
fig.patch.set_facecolor('#090C10')
plt.tight_layout()
plt.savefig('../images/passos_forcados.png', dpi=150, bbox_inches='tight')
plt.show()
print(contagem)